# Run ARC SFT Dataset Construction

Clones the ARC repository from GitLab, installs dependencies, creates `.env` from `.env.example`, and runs the SFT dataset construction helpers on scenario and execution JSONL files.

## 1. Runtime Parameters

Edit these values before running the notebook if needed.

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
SCENARIOS_PATH = PROJECT_DIR / "data" / "mmlu_med_scenarios.jsonl"
EXECUTIONS_PATH = PROJECT_DIR / "data" / "executions.jsonl"
LABELS_PATH = None  # Optional path to valid_strategies.jsonl
OUTPUT_DIR = PROJECT_DIR / "data" / "sft"
PROMPTS_OUTPUT = OUTPUT_DIR / "prompts.jsonl"
PROMPTS_DIR = OUTPUT_DIR / "prompts"
MODE = "sft"  # prompts | labels | sft
PROMPTS_BY_STRATEGY = False
INCLUDE_SELECTION_TARGET = False
SEED = 42
TRAIN_RATIO = 0.8
VALIDATION_RATIO = 0.1
TEST_RATIO = 0.1
FORCE_RECLONE = False


## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-21-jdk-headless git git-lfs wget > /dev/null
!git lfs install


## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")


## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)


## 5. Create and Load `.env`

The repository only tracks `.env.example`. This cell copies it to `.env` when needed.

In [ ]:
import os
import shutil

env_path = PROJECT_DIR / ".env"
env_example_path = PROJECT_DIR / ".env.example"

if not env_path.exists():
    if not env_example_path.exists():
        raise FileNotFoundError(f"Missing both {env_path} and {env_example_path}")
    shutil.copyfile(env_example_path, env_path)
    print(f"Created {env_path} from {env_example_path}")
else:
    print(f"Using existing {env_path}")

def load_dotenv(path):
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_dotenv(env_path)
for key in ["USE_DRIVE", "DRIVE_ROOT", "LOCAL_ROOT", "OUTPUT_DIR", "HF_TOKEN"]:
    print(f"{key}={os.environ.get(key, '')}")


## 6. Mount Google Drive

Drive must be mounted from the notebook kernel, not from the CLI subprocess.

In [ ]:
use_drive = os.environ.get("USE_DRIVE", "false").lower() in {"1", "true", "yes", "y", "on"}
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("USE_DRIVE is false; skipping Google Drive mount.")


## 7. Preflight Check

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))

preflight = subprocess.run(
    [
        sys.executable,
        "-c",
        "from src.sft_dataset import STRATEGY_ORDER; print('python ok'); print(STRATEGY_ORDER)",
    ],
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)
print("--- STDOUT ---")
print(preflight.stdout)
print("--- STDERR ---")
print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")


## 8. Run SFT Construction

In [ ]:
cmd = [sys.executable, "-u", "-m", "src.sft_dataset"]

if MODE == "prompts":
    output_path = PROMPTS_DIR if PROMPTS_BY_STRATEGY else PROMPTS_OUTPUT
    cmd.extend(["prompts", str(SCENARIOS_PATH), str(output_path)])
    if PROMPTS_BY_STRATEGY:
        cmd.append("--by-strategy")
elif MODE == "labels":
    cmd.extend(["labels", str(EXECUTIONS_PATH), str(OUTPUT_DIR / "valid_strategies.jsonl")])
elif MODE == "sft":
    cmd.extend(["sft", str(EXECUTIONS_PATH), str(OUTPUT_DIR)])
    if LABELS_PATH is not None:
        cmd.extend(["--labels", str(LABELS_PATH)])
    cmd.extend([
        "--seed", str(SEED),
        "--train-ratio", str(TRAIN_RATIO),
        "--validation-ratio", str(VALIDATION_RATIO),
        "--test-ratio", str(TEST_RATIO),
    ])
    if INCLUDE_SELECTION_TARGET:
        cmd.append("--selection-target")
else:
    raise ValueError(f"Unknown MODE: {MODE}")

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd,
    cwd=PROJECT_DIR,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"SFT construction failed with exit code {returncode}")


## 9. Inspect Output

In [ ]:
use_drive = os.environ.get("USE_DRIVE", "false").lower() in {"1", "true", "yes", "y", "on"}
root_dir = Path(os.environ["DRIVE_ROOT"] if use_drive else os.environ["LOCAL_ROOT"])
sft_root = root_dir / "outputs" / "sft"

print(f"SFT root: {sft_root}")
if sft_root.exists():
    for path in sorted(sft_root.glob("*.jsonl")):
        print(f"- {path.name}: {path.stat().st_size / 1024:.1f} KiB")
        with path.open("r", encoding="utf-8") as handle:
            first_line = handle.readline().strip()
        if first_line:
            print(first_line[:2000])
else:
    print("No SFT output directory was found.")
